# 🎥 Video 14: 3D Conformer Generation & Geometry Optimization
**RDKit Masterclass — From Zero to Drug Discovery AI**
* **Duration:** ~25 - 30 Minutes
* **Curriculum Level:** LEVEL 8 — Advanced RDKit & 3D Conformer Modeling
* **Target Audience:** Computational Chemists, Molecular Docking Scientists, AI Researchers

---

### ⏱️ Video Agenda & Timestamps
* **00:00 - 04:30** | Why 3D Conformations Matter in Drug Binding & Stereochemistry
* **04:30 - 09:30** | Distance Geometry Embedding: `AllChem.EmbedMolecule` & ETKDG v3
* **09:30 - 14:30** | Molecular Mechanics Energy Minimization: UFF vs MMFF94 / MMFF94s
* **14:30 - 19:30** | Conformer Ensemble Generation: `EmbedMultipleConfs` & Energy Ranking
* **19:30 - 24:00** | Calculating RMSD Matrices & Pruning Redundant Conformers
* **24:00 - 27:30** | 3D Pharmacophore Alignment: `AllChem.AlignMol`
* **27:30 - 30:00** | Hands-On Challenge: Exporting Docking-Ready 3D SDF Files

---

### 🎯 Learning Objectives
1. Generate initial 3D coordinates from 1D SMILES using the modern **ETKDG v3** algorithm.
2. Minimize conformational strain energies using the Merck Molecular Force Field (**MMFF94s**).
3. Sample the conformational landscape by generating multi-conformer ensembles.
4. Calculate pairwise Root-Mean-Square Deviations (RMSD) and cluster conformers.
5. Align 3D flexible ligands against a rigid bioactive reference template.

In [1]:
# Step 0: Imports and Setup
import rdkit
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print(f"RDKit Version: {rdkit.__version__}")

RDKit Version: 2026.03.6


## 1. Why 3D Conformers?
Small drug molecules are flexible: single bonds rotate freely around dihedral angles.
While SMILES describes 2D graph connectivity, a drug binds to its target receptor in a **specific 3D bioactive conformation** (often within $1-3\text{ kcal/mol}$ of the global energy minimum).

To generate realistic 3D structures:
1. **Explicit Hydrogens:** Hydrogens must be added (`Chem.AddHs`) so steric clashes and hydrogen-bonding geometries are physically accurate.
2. **Distance Geometry (ETKDG):** Uses experimental torsion-angle knowledge from the Cambridge Structural Database (CSD) to generate initial 3D embeds.
3. **Force Field Minimization:** Optimizes bond lengths, angles, and torsions using MMFF94 or UFF.

In [2]:
# Generate 3D coordinates for Ibuprofen
ibuprofen_smi = "CC(C)Cc1ccc(cc1)C(C)C(=O)O"
mol_2d = Chem.MolFromSmiles(ibuprofen_smi)

# 1. Add explicit Hydrogens (CRITICAL for 3D modeling)
mol_3d = Chem.AddHs(mol_2d)
print(f"2D Heavy Atoms: {mol_2d.GetNumAtoms()} -> 3D Total Atoms with Hs: {mol_3d.GetNumAtoms()}")

# 2. Embed 3D coordinates using modern ETKDG v3 parameters
params = AllChem.ETKDGv3()
params.randomSeed = 42
embed_status = AllChem.EmbedMolecule(mol_3d, params)

print(f"Embedding success status (0 = success): {embed_status}")

2D Heavy Atoms: 15 -> 3D Total Atoms with Hs: 33
Embedding success status (0 = success): 0


## 2. Molecular Mechanics Energy Minimization (MMFF94s)
The raw distance-geometry embed has geometric strain.
We optimize the geometry using the **Merck Molecular Force Field (MMFF94s)**, designed specifically for drug-like organic molecules and aromatic/heteroaromatic rings.

In [3]:
# Energy minimization using MMFF94s
converged = AllChem.MMFFOptimizeMolecule(mol_3d, mmffVariant="MMFF94s", maxIters=1000)
print(f"Force field minimization converged (0 = converged): {converged}")

# Calculate total potential energy
props = AllChem.MMFFGetMoleculeProperties(mol_3d, mmffVariant="MMFF94s")
ff = AllChem.MMFFGetMoleculeForceField(mol_3d, props)
min_energy = ff.CalcEnergy()

print(f"MMFF94s Potential Energy: {min_energy:.2f} kcal/mol")

Force field minimization converged (0 = converged): 0
MMFF94s Potential Energy: 24.07 kcal/mol


## 3. Conformer Ensemble Generation (`EmbedMultipleConfs`)
Flexible molecules have multiple low-energy conformers.
Let's generate an ensemble of 20 distinct 3D conformers and rank them by relative energy.

In [4]:
# Generate 20 conformers for Ibuprofen
embed_params = AllChem.ETKDGv3()
embed_params.pruneRmsThresh = 0.5  # Discard conformers with RMSD < 0.5 Angstrom to avoid duplicates
embed_params.randomSeed = 42

cids = AllChem.EmbedMultipleConfs(mol_3d, 20, embed_params)

print(f"Generated {len(cids)} diverse conformers (after 0.5 Å RMSD pruning).")

# Minimize each conformer and record energies
conformer_energies = []
for cid in cids:
    AllChem.MMFFOptimizeMolecule(mol_3d, confId=cid, mmffVariant="MMFF94s")
    ff = AllChem.MMFFGetMoleculeForceField(mol_3d, props, confId=cid)
    energy = ff.CalcEnergy()
    conformer_energies.append((cid, energy))

# Sort by energy ascending
conformer_energies.sort(key=lambda x: x[1])
lowest_energy = conformer_energies[0][1]

df_confs = pd.DataFrame([
    {"Conf_ID": cid, "Absolute_Energy_kcal_mol": round(e, 2), "Delta_E_kcal_mol": round(e - lowest_energy, 2)}
    for cid, e in conformer_energies
])

print("=== 🔬 TOP 5 LOWEST ENERGY CONFORMERS ===")
display(df_confs.head(5))

Generated 8 diverse conformers (after 0.5 Å RMSD pruning).
=== 🔬 TOP 5 LOWEST ENERGY CONFORMERS ===


,Conf_ID,Absolute_Energy_kcal_mol,Delta_E_kcal_mol
0,6,23.79,0.00
1,4,23.79,0.00
2,3,23.79,0.00
3,2,24.07,0.28
4,0,24.07,0.28


## 4. Pairwise RMSD Matrix Calculation
We can evaluate how geometrically distinct the conformers are by calculating their pairwise Root-Mean-Square Deviation (RMSD).

In [5]:
# Calculate RMSD matrix relative to the lowest energy conformer (Conf 0)
ref_cid = conformer_energies[0][0]
rms_list = []

for cid, _ in conformer_energies:
    rms = AllChem.GetConformerRMS(mol_3d, ref_cid, cid, prealigned=False)
    rms_list.append(rms)

df_confs["RMSD_to_Best"] = [round(r, 3) for r in rms_list]
display(df_confs.head(5))

,Conf_ID,Absolute_Energy_kcal_mol,Delta_E_kcal_mol,RMSD_to_Best
0,6,23.79,0.00,0.000
1,4,23.79,0.00,1.879
2,3,23.79,0.00,2.321
3,2,24.07,0.28,2.112
4,0,24.07,0.28,1.925


## 5. 3D Molecular Alignment (`AllChem.AlignMol`)
In pharmacophore modeling, you frequently need to align candidate molecules onto a bioactive reference conformation.

In [6]:
# Align conformer 1 onto conformer 0 based on heavy atoms
conf_a = conformer_energies[0][0]
conf_b = conformer_energies[1][0]

rmsd_val = AllChem.AlignMol(mol_3d, mol_3d, prbCid=conf_b, refCid=conf_a)
print(f"Alignment RMSD between Conf {conf_a} and Conf {conf_b}: {rmsd_val:.3f} Å")

Alignment RMSD between Conf 6 and Conf 4: 1.879 Å


## 6. 🎯 Video Hands-On Challenge & Solution
### Problem:
1. Export the top 3 lowest-energy 3D conformers of Ibuprofen into an SDF file `ibuprofen_3d_ensemble.sdf`.
2. Attach the relative energy `Delta_E_kcal_mol` as an SD property tag to each conformer.
3. Re-load the SDF and verify that 3 distinct 3D conformers exist.

In [7]:
out_3d_sdf = "ibuprofen_3d_ensemble.sdf"
writer = Chem.SDWriter(out_3d_sdf)

for rank, (cid, e) in enumerate(conformer_energies[:3], 1):
    delta_e = e - lowest_energy
    mol_3d.SetProp("_Name", f"Ibuprofen_Conf_{rank}")
    mol_3d.SetDoubleProp("Delta_E_kcal_mol", round(delta_e, 3))
    mol_3d.SetIntProp("Conformer_ID", cid)
    writer.write(mol_3d, confId=cid)

writer.close()

# Verify
suppl_3d = Chem.SDMolSupplier(out_3d_sdf)
print(f"✅ Verified: Loaded {len(suppl_3d)} conformers from '{out_3d_sdf}'")
for m in suppl_3d:
    print(f"  {m.GetProp('_Name')}: Delta_E = {m.GetProp('Delta_E_kcal_mol')} kcal/mol, 3D Conformer count = {m.GetNumConformers()}")

✅ Verified: Loaded 3 conformers from 'ibuprofen_3d_ensemble.sdf'
  Ibuprofen_Conf_1: Delta_E = 0 kcal/mol, 3D Conformer count = 1
  Ibuprofen_Conf_2: Delta_E = 0 kcal/mol, 3D Conformer count = 1
  Ibuprofen_Conf_3: Delta_E = 0.001 kcal/mol, 3D Conformer count = 1


## 7. Summary & Key Takeaways
* Always add explicit hydrogens (`Chem.AddHs`) before generating 3D coordinates.
* **ETKDG v3** embeds realistic conformations using experimental torsion rules.
* **MMFF94s** minimizes conformational strain energies for drug-like molecules.
* Conformer ensembles with RMSD pruning capture the flexible 3D shape space for molecular docking and pharmacophore modeling.

**Next Up in Video 15:** **Machine Learning QSAR & Modern AI Representations** — scaffold-based splitting, predicting bioactivity ($pIC_{50}$), and transitioning to molecular graphs!